## Notebook 05 - PySpark Avancé et Optimisation

**Objectifs** :
- Analyser les plans d'exécution Spark (`.explain()`)
- Optimiser la table Delta SIRENE (`OPTIMIZE`, `ZORDER`, `VACUUM`)
- Comprendre AQE (Adaptive Query Execution)
- Maîtriser `repartition` vs `coalesce`
- Caching stratégique avec `persist()`
- Comparer UDF vs `pandas_udf` en termes de performance

**Table source** : `sirene_clean_delta` - 134 666 lignes, 28 colonnes, partitionné par `categorie_entreprise`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType, FloatType, LongType
from pyspark.sql.functions import udf, pandas_udf
import time

# Constantes de chemin (Unity Catalog Volumes) 
DELTA_CLEAN   = "/Volumes/workspace/default/raw_data/sirene_clean_delta"
DELTA_ANALYTIQUE = "/Volumes/workspace/default/raw_data/sirene_analytique_delta"

# Chargement des tables Delta 
df_clean = spark.read.format("delta").load(DELTA_CLEAN)
df_analytique = spark.read.format("delta").load(DELTA_ANALYTIQUE)

# Vérification rapide (lazy eval - ne lit pas les données encore) 
print(f"Schéma df_clean : {len(df_clean.columns)} colonnes")
print(f"Schéma df_analyt : {len(df_analytique.columns)} colonnes")

# count() = action → déclenche le scan réel
print(f"df_clean : {df_clean.count():,} lignes")
print(f"df_analytique : {df_analytique.count():,} lignes")

Schéma df_clean : 28 colonnes
Schéma df_analyt : 11 colonnes
df_clean : 134,666 lignes
df_analytique : 680 lignes


In [0]:
# Plan à analyser : filtre + agrégation
df_sample = df_clean.filter(
  F.col("categorie_entreprise") == "PME"
).groupBy("code_commune").agg(
  F.count("*").alias("nb_etablissements"),
  F.countDistinct("siren").alias("nb_entreprises")
)

print("MODE SIMPLE (par défaut)")
df_sample.explain()

print("\nMODE FORMATTED (le plus lisible - utiliser en entretien)")
df_sample.explain(mode="formatted")

print("\nMODE COST (avec statistiques de cardinalité estimées)")
df_sample.explain(mode="cost")

# Note : mode="extended" affiche plan logique non-analysé + analysé + physique

MODE SIMPLE (par défaut)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[code_commune#18504], functions=[finalmerge_count(merge count#18599L) AS count(1)#18592L, finalmerge_count(distinct merge count#18601L) AS count(siren)#18593L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#24654]
               +- PhotonShuffleExchangeSink hashpartitioning(code_commune#18504, 16)
                  +- PhotonGroupingAgg(keys=[code_commune#18504], functions=[merge_count(merge count#18599L) AS count#18599L, partial_count(distinct siren#18494) AS count#18601L])
                     +- PhotonGroupingAgg(keys=[code_commune#18504, siren#18494], functions=[merge_count(merge count#18599L) AS count#18599L])
                        +- PhotonShuffleExchangeSource
                           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#24646]

In [0]:
# Cas 1 : filtre sur colonne de PARTITION → PartitionFilters 
# Delta n'ouvre que les fichiers de la partition PME
print("Filtre sur colonne de partition (categorie_entreprise) :")
df_clean.filter(F.col("categorie_entreprise") == "PME").explain(mode="formatted")

# Cas 2 : filtre sur colonne ordinaire → PushedFilters 
# Delta utilise les statistiques min/max du _delta_log pour sauter des fichiers
print("\nFiltre sur colonne ordinaire (etat_admin_etab) :")
df_clean.filter(F.col("etat_admin_etab") == "Actif").explain(mode="formatted")

# Cas 3 : colonnes non sélectionnées → Column Pruning 
# Spark ne lit que les colonnes effectivement utilisées (Parquet est columnar)
print("\nColumn pruning - lecture de 3 colonnes seulement :")
df_clean.select("siret", "code_commune", "categorie_entreprise") \
    .filter(F.col("categorie_entreprise") == "GE") \
    .explain(mode="formatted")
# Chercher : ReadSchema: struct

Filtre sur colonne de partition (categorie_entreprise) :
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [28]: [siren#18494, nic#18495, siret#18496L, statut_diffusion#18497, date_creation_etab#18498, tranche_effectif#18499, activite_principale_etab#18500, etablissement_siege#18501, code_postal#18502, commune#18503, code_commune#18504, code_departement#18505, departement#18506, code_region#18507, region#18508, etat_admin_etab#18509, date_fermeture_etab#18510, denomination_unite_legale#18511, etat_admin_ul#18513, caractere_employeur#18514, activite_principale_ul#18515, categorie_juridique#18516, date_creation_ul#18517, loaded_at#18518, siret_calcule#18519, est_siege#18520, source_pipeline#18521, categorie_entreprise#18512]
Location: PreparedDeltaFileIndex [dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta]
PartitionFilters: [isnotnull(categorie_entreprise#18512), (cate

In [0]:
# DESCRIBE DETAIL : métriques de la table Delta 
spark.sql(f"""
  DESCRIBE DETAIL delta.`{DELTA_CLEAN}`
""").select(
  "format", "numFiles", "sizeInBytes"
).display()

# Compter le nombre de partitions réelles (valeurs distinctes)
print(f"Nombre de partitions réelles : {df_clean.select('categorie_entreprise').distinct().count()}")
df_clean.groupBy("categorie_entreprise").count().orderBy("count", ascending=False).display()

# DESCRIBE HISTORY : journal des opérations (versions 0→3 depuis S2) 
# Version 0 = WRITE initial | 1 = MERGE upsert | 2 = WRITE append | 3 = re-run
spark.sql(f"""
  DESCRIBE HISTORY delta.`{DELTA_CLEAN}`
""").select(
  "version", "timestamp", "operation", "operationParameters"
).display()

format,numFiles,sizeInBytes
delta,4,5653869


Nombre de partitions réelles : 4


categorie_entreprise,count
PME,79202
INCONNUE,47029
ETI,5039
GE,3396


version,timestamp,operation,operationParameters
16,2026-07-10T08:44:06.000Z,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [""code_commune"",""activite_principale_etab""], batchId -> 0)"
15,2026-07-09T10:40:16.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)"
14,2026-07-09T10:40:03.000Z,MERGE,"Map(predicate -> [""(siret#25501L = siret#25528L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])"
13,2026-07-09T10:39:48.000Z,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""categorie_entreprise""], canOverwriteSchema -> true)"
12,2026-07-09T10:30:19.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)"
11,2026-07-09T10:30:02.000Z,MERGE,"Map(predicate -> [""(siret#14255L = siret#14282L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])"
10,2026-07-09T10:29:39.000Z,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""categorie_entreprise""], canOverwriteSchema -> true)"
9,2026-07-09T09:42:42.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)"
8,2026-07-09T09:42:29.000Z,MERGE,"Map(predicate -> [""(siret#16777L = siret#16804L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])"
7,2026-07-09T09:42:14.000Z,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""categorie_entreprise""], canOverwriteSchema -> true)"


In [0]:
# OPTIMIZE : compaction des fichiers Parquet 
# Fusionne les petits fichiers en fichiers de ~256 MB (default)
# ZORDER BY : colocalisation selon les colonnes filtrées le plus souvent
# Ici : code_commune (agrégations analytiques) + activite_principale_etab (secteur NAF)
result = spark.sql(f"""
  OPTIMIZE delta.`{DELTA_CLEAN}`
  ZORDER BY (code_commune, activite_principale_etab)
""")
result.display()
# Colonnes de résultat : numFilesAdded, numFilesRemoved, filesAdded, filesRemoved

# Vérifier l'impact : numFiles doit avoir diminué 
spark.sql(f"""
  DESCRIBE DETAIL delta.`{DELTA_CLEAN}`
""").select("numFiles", "sizeInBytes").display()

# Nouvelle version dans HISTORY 
spark.sql(f"""
  DESCRIBE HISTORY delta.`{DELTA_CLEAN}` LIMIT 3
""").select("version", "timestamp", "operation").display()

path,metrics
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 3, List(minCubeSize(107374182400), List(1, 2074666), List(3, 3579203), 1, List(0, 0), 0, null), null, 0, 0, 4, 4, false, 0, 0, 1783676057501, 1783676058726, 8, 0, null, List(0, 0), null, 28, 28, 0, 0, null, null)"


numFiles,sizeInBytes
4,5653869


version,timestamp,operation
16,2026-07-10T08:44:06.000Z,OPTIMIZE
15,2026-07-09T10:40:16.000Z,WRITE
14,2026-07-09T10:40:03.000Z,MERGE


In [0]:
# DRY RUN : voir les fichiers qui seraient supprimés (sans supprimer) 
print("DRY RUN :")
spark.sql(f"""
  VACUUM delta.`{DELTA_CLEAN}` RETAIN 168 HOURS DRY RUN
""").display()

# VACUUM réel 
# ATTENTION : irréversible - les versions antérieures à 7 jours ne seront plus accessibles
# Pour nos données de formation (table créée récemment), DRY RUN retournera 0 fichier

# Dans un vrai pipeline mensuel, VACUUM est crucial pour éviter l'accumulation :
spark.sql(f"""
  VACUUM delta.`{DELTA_CLEAN}` RETAIN 168 HOURS
""")
print("VACUUM terminé.")

# Bonne pratique : aussi OPTIMIZE + VACUUM sur la table analytique 
spark.sql(f"""
  OPTIMIZE delta.`{DELTA_ANALYTIQUE}`
  ZORDER BY (code_commune, libelle_secteur)
""").display()

DRY RUN :


path


VACUUM terminé.


path,metrics
dbfs:/Volumes/workspace/default/raw_data/sirene_analytique_delta,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 1, List(minCubeSize(107374182400), List(0, 0), List(1, 11396), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1783676075369, 1783676076084, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null)"


In [0]:
# AQE est toujours actif en Serverless Databricks (géré automatiquement)
# Les configs AQE ne sont pas accessibles via spark.conf.get() en Serverless
print("AQE est actif par défaut en Serverless Databricks")
print("Les 3 mécanismes AQE actifs :")
print(" 1. Coalescing post-shuffle (fusionne partitions vides/petites)")
print(" 2. Conversion SortMerge → Broadcast (si table devient petite)")
print(" 3. Gestion des skews (divise les partitions trop grosses)")

# Les 3 mécanismes AQE 
# 1. Coalescing post-shuffle : fusionne les partitions vides/petites après un Exchange
#  → évite des milliers de tâches pour 200 petites partitions
# 2. Conversion SortMergeJoin → BroadcastHashJoin si une table devient petite
#  → après filtrage, la table est peut-être assez petite pour être broadcastée
# 3. Gestion des skews : divise les partitions trop grosses en sous-partitions
#  → évite qu'1 partition de 10GB bloque 199 partitions de 1MB

AQE est actif par défaut en Serverless Databricks
Les 3 mécanismes AQE actifs :
 1. Coalescing post-shuffle (fusionne partitions vides/petites)
 2. Conversion SortMerge → Broadcast (si table devient petite)
 3. Gestion des skews (divise les partitions trop grosses)


In [0]:
# Requête avec shuffle (GROUP BY sur une colonne non-partition)
df_agg = df_clean.groupBy("activite_principale_etab", "categorie_entreprise") \
         .agg(
           F.count("*").alias("nb_etablissements"),
           F.countDistinct("siren").alias("nb_entreprises"),
           F.sum(
             F.when(F.col("etablissement_siege") == "oui", 1).otherwise(0)
           ).alias("nb_sieges")
         )

# Avant exécution : AQE prépare un plan préliminaire 
print("AVANT l'action (plan préliminaire) :")
df_agg.explain(mode="formatted")
# Chercher : AdaptiveSparkPlan isFinalPlan=false

# Déclencher l'exécution (action) 
nb = df_agg.count()
print(f"\nNombre de combinaisons code_naf x categorie : {nb}")

# Après exécution : AQE a optimisé le plan réel 
print("\nAPRÈS l'action (plan final optimisé) :")
df_agg.explain(mode="formatted")
# Chercher : AdaptiveSparkPlan isFinalPlan=true
# AQE a peut-être : fusionné des partitions, changé le type de join, etc.

df_agg.display()

AVANT l'action (plan préliminaire) :
== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonGroupingAgg (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonGroupingAgg (6)
                        +- PhotonShuffleExchangeSource (5)
                           +- PhotonShuffleMapStage (4)
                              +- PhotonShuffleExchangeSink (3)
                                 +- PhotonGroupingAgg (2)
                                    +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [4]: [siren#18494, activite_principale_etab#18500, etablissement_siege#18501, categorie_entreprise#18512]
Location: PreparedDeltaFileIndex [dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta]
ReadSchema: struct<siren:int,activite_principale_etab

activite_principale_etab,categorie_entreprise,nb_etablissements,nb_entreprises,nb_sieges
9601BR,PME,26,19,15
4932ZA,PME,148,148,148
2829BZ,PME,5,5,5
4311ZZ,PME,5,5,5
1072ZZ,PME,10,10,9
4722ZA,PME,25,24,22
9609ZP,PME,32,32,32
3102ZZ,PME,3,3,3
3319ZZ,PME,6,6,6
3320CZ,PME,1,1,1


In [0]:
# Note : en Serverless, les partitions sont gérées automatiquement
# L'API RDD (.rdd.getNumPartitions()) n'est pas disponible
# Mais repartition() et coalesce() fonctionnent normalement

# repartition(n) : shuffle complet, équilibré 
# Utiliser pour : augmenter le parallélisme, rééquilibrer après un filtre
print("repartition(8) - shuffle complet :")
df_rep8 = df_clean.repartition(8)
df_rep8.explain() # Chercher : Exchange RoundRobinPartitioning(8)

# coalesce(n) : pas de shuffle, diminue uniquement 
# Utiliser pour : réduire le nombre de fichiers à l'écriture (avant df.write)
print("\ncoalesce(2) - pas de shuffle :")
df_coal2 = df_clean.coalesce(2)
df_coal2.explain() # PAS de Exchange - coalesce ne shuffle pas

# repartition par colonne : co-location pour les joins 
# Utile avant un join sur code_commune : co-localise les lignes par commune
print("\nrepartition('categorie_entreprise') - co-location :")
df_rep_col = df_clean.repartition("categorie_entreprise")
df_rep_col.explain() # Exchange hashpartitioning(categorie_entreprise)

# Résumé des règles 
print("""
Règles pratiques :
 repartition(n)     → avant un JOIN ou GROUP BY intensif
 coalesce(n)       → avant df.write() pour limiter le nombre de fichiers
 repartition("col")   → avant un JOIN sur cette colonne (co-location)
 Laisser AQE décider   → en général, ne pas forcer le partitionnement
""")

repartition(8) - shuffle complet :
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonShuffleExchangeSource
         +- PhotonShuffleMapStage REPARTITION_BY_NUM, [id=#31802]
            +- PhotonShuffleExchangeSink RoundRobinPartitioning(8)
               +- PhotonSort [siren#18494 ASC NULLS FIRST, nic#18495 ASC NULLS FIRST, siret#18496L ASC NULLS FIRST, statut_diffusion#18497 ASC NULLS FIRST, date_creation_etab#18498 ASC NULLS FIRST, tranche_effectif#18499 ASC NULLS FIRST, activite_principale_etab#18500 ASC NULLS FIRST, etablissement_siege#18501 ASC NULLS FIRST, code_postal#18502 ASC NULLS FIRST, commune#18503 ASC NULLS FIRST, code_commune#18504 ASC NULLS FIRST, code_departement#18505 ASC NULLS FIRST, departement#18506 ASC NULLS FIRST, code_region#18507 ASC NULLS FIRST, region#18508 ASC NULLS FIRST, etat_admin_etab#18509 ASC NULLS FIRST, date_fermeture_etab#18510 ASC NULLS FIRST, denomination_unit

In [0]:
%skip
# ⚠️ CACHE / PERSIST : NON SUPPORTÉ EN SERVERLESS - CETTE CELLULE NE PEUT PAS S'EXÉCUTER
# En Serverless, le caching est géré automatiquement par Spark Connect

from pyspark import StorageLevel

# Cas 1 : SANS cache - chaque action relit depuis Delta 
df_actifs = df_clean.filter(F.col("etat_admin_etab") == "Actif")

t0 = time.time()
n1 = df_actifs.filter(F.col("categorie_entreprise") == "PME").count()
t1 = time.time()
n2 = df_actifs.filter(F.col("categorie_entreprise") == "GE").count()
t2 = time.time()
print(f"Sans cache : PME={n1} ({t1-t0:.2f}s) | GE={n2} ({t2-t1:.2f}s)")

# Cas 2 : AVEC cache() - matérialise en mémoire après la 1ère action 
df_actifs_c = df_clean.filter(F.col("etat_admin_etab") == "Actif")
df_actifs_c.cache()

t0 = time.time()
n1 = df_actifs_c.filter(F.col("categorie_entreprise") == "PME").count() # lit + cache
t1 = time.time()
n2 = df_actifs_c.filter(F.col("categorie_entreprise") == "GE").count()  # lit depuis cache
t2 = time.time()
print(f"Avec cache : PME={n1} ({t1-t0:.2f}s fill) | GE={n2} ({t2-t1:.2f}s hit)")

# Libérer le cache (important pour éviter OOM)
df_actifs_c.unpersist()

# Cas 3 : persist() avec StorageLevel explicite 
# MEMORY_ONLY : plus rapide, peut provoquer recompute si OOM
# MEMORY_AND_DISK : sécurisé, revers sur disque si OOM (≈ cache())
# DISK_ONLY : économique en RAM, mais lecture disque
df_communes = df_clean.select("code_commune", "commune", "departement").distinct()
df_communes.persist(StorageLevel.MEMORY_ONLY)
t0 = time.time()
n_communes = df_communes.count() # matérialise
print(f"\npersist MEMORY_ONLY : {n_communes} communes distinctes ({time.time()-t0:.2f}s)")

# 2ème accès : depuis la cache mémoire
t0 = time.time()
df_communes.filter(F.col("departement") == "Loire-Atlantique").count()
print(f"2ème accès (depuis cache) : {time.time()-t0:.4f}s")

# Libérer
df_communes.unpersist()

print("""
Quand utiliser le cache ?
 ✓ DataFrame réutilisé 2+ fois dans le même notebook
 ✓ Après un filtre lourd (read+filter coûteux) - matérialiser le résultat
 ✓ Avant plusieurs agrégations sur les mêmes données
 ✗ Pas pour des DataFrames lus une seule fois (coût de mise en cache inutile)
 ✗ Pas sur des tables Delta fraîches si elles changent souvent
""")

In [0]:
import pandas as pd

# Fonction Python pure (mapping tranche_effectif → libellé) 
# Contexte : dans df_clean, tranche_effectif est le code INSEE des tranches d'effectif
# 00=0 salarié, 01=1-2, 02=3-5, 11=10-19, 21=20-49, 31=50-99, etc.
TRANCHE_MAP = {
    "00": "0 salarié",
    "01": "1 à 2",
    "02": "3 à 5",
    "03": "6 à 9",
    "11": "10 à 19",
    "12": "20 à 49",
    "21": "50 à 99",
    "22": "100 à 199",
    "31": "200 à 249",
    "32": "250 à 499",
    "41": "500 à 999",
    "42": "1 000 à 1 999",
    "51": "2 000 à 4 999",
    "52": "5 000 à 9 999",
    "53": "10 000 et plus",
}

# Python UDF - sérialisation ROW par ROW 
@udf(returnType=StringType())
def libelle_tranche_udf(code: str) -> str:
    return TRANCHE_MAP.get(code, "Non renseigné")

# Mesure sur 50 000 lignes
# Note : .cache() non disponible en Serverless - mesure directe
df_bench = df_clean.select("siret", "tranche_effectif").limit(50000)

t0 = time.time()
df_bench.withColumn("libelle_udf", libelle_tranche_udf(F.col("tranche_effectif"))).count()
t_udf = time.time() - t0
print(f"Python UDF (classique) : {t_udf:.2f}s")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Python UDF (classique) : 0.71s


In [0]:
# pandas_udf - vectorisée : traite des pd.Series (batches Arrow) 
# Le décorateur @pandas_udf indique le type de retour
@pandas_udf(StringType())
def libelle_tranche_pudf(codes: pd.Series) -> pd.Series:
    # codes est une pd.Series (batch de valeurs, pas une seule valeur)
    return codes.map(lambda x: TRANCHE_MAP.get(x, "Non renseigné"))

t0 = time.time()
df_bench.withColumn("libelle_pudf", libelle_tranche_pudf(F.col("tranche_effectif"))).count()
t_pudf = time.time() - t0
print(f"pandas_udf (vectorisée)  : {t_pudf:.2f}s")

pandas_udf (vectorisée)  : 0.65s


In [0]:
# Approche native : F.when / F.otherwise (pas d'UDF du tout) 
t0 = time.time()
df_bench.withColumn(
    "libelle_native",
    F.when(F.col("tranche_effectif") == "00", "0 salarié")
     .when(F.col("tranche_effectif") == "01", "1 à 2")
     .when(F.col("tranche_effectif") == "02", "3 à 5")
     .when(F.col("tranche_effectif") == "03", "6 à 9")
     .when(F.col("tranche_effectif") == "11", "10 à 19")
     .when(F.col("tranche_effectif") == "12", "20 à 49")
     .when(F.col("tranche_effectif") == "21", "50 à 99")
     .when(F.col("tranche_effectif") == "22", "100 à 199")
     .when(F.col("tranche_effectif").isin("31","32"), "200 à 499")
     .when(F.col("tranche_effectif").isin("41","42"), "500 à 1 999")
     .when(F.col("tranche_effectif").isin("51","52","53"), "2 000 et plus")
     .otherwise("Non renseigné")
).count()
t_native = time.time() - t0

# Approche native : create_map (dictionnaire en SQL natif) 
map_expr = F.create_map([
    item for kv in TRANCHE_MAP.items() for item in (F.lit(kv[0]), F.lit(kv[1]))
])

t0 = time.time()
df_bench.withColumn(
    "libelle_map",
    F.coalesce(map_expr[F.col("tranche_effectif")], F.lit("Non renseigné"))
).count()
t_map = time.time() - t0

# Résumé benchmark 
print(f"\n{'=' * 55}")
print(f"  BENCHMARK sur {50_000:,} lignes")
print(f"{'=' * 55}")
print(f"  Python UDF        : {t_udf:.2f}s   (lent - sérialisation JVM↔Python)")
print(f"  pandas_udf Arrow  : {t_pudf:.2f}s   (rapide - vectorisé)")
print(f"  F.when/otherwise  : {t_native:.2f}s   (très rapide - JVM pur)")
print(f"  F.create_map      : {t_map:.2f}s   (très rapide - JVM pur)")
print(f"{'=' * 55}")
print(f"  Speedup pudf/udf  : {t_udf/t_pudf:.1f}x")
print(f"  Speedup native/udf: {t_udf/t_native:.1f}x")
print("""
Règle d'or :
  1. Fonctions Spark natives (F.col, F.when, F.lit, etc.)  → TOUJOURS préférer
  2. pandas_udf                                            → logique Python complexe
  3. Python UDF classique                                  → éviter en production
""")

# Note : .unpersist() non supporté en Serverless
# df_bench.unpersist()


  BENCHMARK sur 50,000 lignes
  Python UDF        : 0.71s   (lent - sérialisation JVM↔Python)
  pandas_udf Arrow  : 0.65s   (rapide - vectorisé)
  F.when/otherwise  : 0.81s   (très rapide - JVM pur)
  F.create_map      : 0.66s   (très rapide - JVM pur)
  Speedup pudf/udf  : 1.1x
  Speedup native/udf: 0.9x

Règle d'or :
  1. Fonctions Spark natives (F.col, F.when, F.lit, etc.)  → TOUJOURS préférer
  2. pandas_udf                                            → logique Python complexe
  3. Python UDF classique                                  → éviter en production



In [0]:
# DataFrame de référence (cache - réutilisé 3 fois ci-dessous) 
df_actifs = df_clean.filter(F.col("etat_admin_etab") == "Actif") \
                    .select(
                        "siret", "siren", "code_commune", "commune",
                        "departement", "code_departement", "categorie_entreprise",
                        "activite_principale_etab", "tranche_effectif",
                        "etablissement_siege", "caractere_employeur"
                    )
# Note : .cache() non supporté en Serverless
df_actifs.count()  # matérialise (1ère lecture depuis Delta)

# libellé tranche via F.create_map (natif) 
map_tranche = F.create_map([
    item for kv in TRANCHE_MAP.items() for item in (F.lit(kv[0]), F.lit(kv[1]))
])

# Agrégation par commune x secteur x taille 
df_optimised = (
    df_actifs
    .withColumn("libelle_tranche",
        F.coalesce(map_tranche[F.col("tranche_effectif")], F.lit("Non renseigné")))
    .withColumn("est_employeur",
        F.when(F.col("caractere_employeur") == "O", True).otherwise(False))
    .groupBy("code_commune", "commune", "categorie_entreprise", "libelle_tranche")
    .agg(
        F.count("*").alias("nb_etablissements"),
        F.countDistinct("siren").alias("nb_entreprises"),
        F.sum(F.when(F.col("etablissement_siege") == "oui", 1).otherwise(0)).alias("nb_sieges"),
        F.sum(F.when(F.col("est_employeur"), 1).otherwise(0)).alias("nb_employeurs")
    )
    .withColumn("pct_sieges",
        F.round(F.col("nb_sieges") / F.col("nb_etablissements") * 100, 1))
    .orderBy("code_commune", "categorie_entreprise", "libelle_tranche")
)

# Plan optimisé (AQE, partition pruning, column pruning, etc.)
df_optimised.explain(mode="formatted")
print(f"\nLignes résultat : {df_optimised.count():,}")
df_optimised.display()

# Note : .unpersist() non supporté en Serverless
# df_actifs.unpersist()

== Physical Plan ==
AdaptiveSparkPlan (20)
+- == Initial Plan ==
   PhotonResultStage (19)
   +- PhotonColumnarToRow (18)
      +- PhotonSort (17)
         +- PhotonShuffleExchangeSource (16)
            +- PhotonShuffleMapStage (15)
               +- PhotonShuffleExchangeSink (14)
                  +- PhotonProject (13)
                     +- PhotonGroupingAgg (12)
                        +- PhotonShuffleExchangeSource (11)
                           +- PhotonShuffleMapStage (10)
                              +- PhotonShuffleExchangeSink (9)
                                 +- PhotonGroupingAgg (8)
                                    +- PhotonGroupingAgg (7)
                                       +- PhotonShuffleExchangeSource (6)
                                          +- PhotonShuffleMapStage (5)
                                             +- PhotonShuffleExchangeSink (4)
                                                +- PhotonGroupingAgg (3)
                                   

code_commune,commune,categorie_entreprise,libelle_tranche,nb_etablissements,nb_entreprises,nb_sieges,nb_employeurs,pct_sieges
44009,BASSE-GOULAINE,ETI,Non renseigné,99,98,51,0,51.5
44009,BASSE-GOULAINE,GE,Non renseigné,26,24,1,0,3.8
44009,BASSE-GOULAINE,INCONNUE,Non renseigné,582,577,560,0,96.2
44009,BASSE-GOULAINE,PME,Non renseigné,997,982,906,0,90.9
44018,BOUAYE,ETI,Non renseigné,17,17,3,0,17.6
44018,BOUAYE,GE,Non renseigné,19,17,1,0,5.3
44018,BOUAYE,INCONNUE,Non renseigné,424,418,407,0,96.0
44018,BOUAYE,PME,Non renseigné,785,768,714,0,91.0
44020,BOUGUENAIS,ETI,Non renseigné,167,129,46,0,27.5
44020,BOUGUENAIS,GE,Non renseigné,104,77,11,0,10.6


## Récapitulatif des optimisations

| Technique | Quand | Impact | Support Serverless |
|-----------|-------|--------|-------------------|
|`.explain(mode="formatted")` | Diagnostiquer un plan lent | Identifier les shuffles, missing pushdowns | Oui |
|`OPTIMIZE + ZORDER BY` | Après ingestion / MERGE | Réduit le nombre de fichiers, améliore les scans | Oui |
|`VACUUM RETAIN 168 HOURS` | En production, 1x/semaine | Libère l'espace disque | Oui |
|AQE (`isFinalPlan=false→true`) | Automatique (Databricks serverless) | Optimisation adaptative sans intervention | Oui |
|`coalesce(n)` avant `.write()` | Écriture avec peu de fichiers | Réduit le small files problem | Oui |
|`df.cache()` / `persist()` | DF réutilisé 2+ fois | Évite les relectures Delta répétées | Non (géré automatiquement, non supporté explicitement) |
|`pandas_udf` | Logique Python complexe | 5-20x plus rapide qu'une Python UDF classique | Oui |
|**Fonctions natives Spark** | **Toujours si possible** | **Pas de sérialisation JVM↔Python** | Oui |
 